# Getting started

## Overview

This Jupyter notebook allows the specification of the run mode and various analysis parameters. 

## Run modes

The notebook can be run in two modes: 
1. **Single file processing**
   - The notebook will only process one specified file and will load the original and the calculated pixel variance stacks in the Napari viewer for visual inspection. Results will be saved automatically in an "analysis" folder which will be created in the indicated directory. 
2. **Batch processing**
   - The notebook will process all tiff or ome.tiff files located in the indicated input folder. The Napari viewer will not be used to free up memory after every processed file. Results will be saved automatically in an "analysis" folder which will be created in the indicated directory.
  
## Input parameters and options

1. Directory
   - The directory in which the file(s) to be processed is/are located. 
2. Checkbox option for batch processing
   - Tick the box to process all ome.tiff/tiff files located within the indicated directory. Batch mode is the defaultrun mode.
   - If the box is not ticked, a file located within the directory folder needs to be specified. Copy and past the file name including the file extension.
3. File name
   - Only required when not batch processing
4. Video rate (fps)
   - In batch mode, all files will be processed using this video rate. Ensure that the directory only contains files recorded with the same frame rate.
5. Beat Rate (Hz)
   - In batch mode, all files will be processed using this beat rate. Ensure that the directory only contains images where the sample exhibits the same beat rate.
6. Offset (frames)
   - Add an integer value to manually specify the offset number of frames. Be sure to untick the Checkbox "Calculate offset from video rate & beat rate" otherwise this number will be overwritten.
7. Checkbox option for calculating the offset from video rate & beat rate
   - The calculation follows the formula: $$ offset = fractional offset * (video rate (fps) / beat rate (Hz)) $$
   - Fractional offset: set a fraction between 0 and 1.
8. Checkbox option to calibrate the calculated speed/contraction values with a pixel calibration.
   - If the pixel calibration, e.g. 1.38 micron/pixel, is known, enter that value in the box below.
   - The calculated speed and contraction values will then be scaled accordingly. 
   - If the pixel calibration is not known, untick the box and the generated results will be in px/s and px for speed and contraction respectively.
9. Pixel calibration.
   - The expected unit is micron.
  
Once all parameters are set, click on the "Submit Parameters" button and move on to the next code cell. 

In [1]:
# ---- Import utility functions ----
import napari
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from scipy.fft import fft, fftshift, ifft
from scipy.signal import windows
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---- User input dialog ---- 
# Input widgets
input_directory_input = widgets.Text(
    value='//magpie.liv.ac.uk/lab-rabb/rabb/2025-10-24_Killifish/Videos/Tiff_stacks',
    #value='C:/Users/directory',
    description='Directory:',
    style={'description_width': 'auto'},
    layout=widgets.Layout(width='600px')
)

checkbox_batch_processing = widgets.Checkbox(
    value=True,
    description='Process all ome.tiff, tiff files in directory above?',
    indent=False,
    style={'description_width': 'auto'}
)

file_name_info_message = widgets.HTML(
    value='<i>A filename is only required when the batch option is not ticked.</i>',
)

input_file_input = widgets.Text(
    value='16-1-M_0001-0375.tif',
    #value='time-lapse-file.ome.tif',
    description='File name:',
    style={'description_width': 'auto'},
    layout=widgets.Layout(width='400px')
)

video_rate_input = widgets.IntText(
    value=34,
    description='Video rate (fps):',
    style={'description_width': 'auto'}
)

exp_beat_rate_input = widgets.FloatText(
    value=1.125,
    description='Beat rate (Hz):',
    style={'description_width': 'auto'}
)

offset_input = widgets.FloatText(
    value=15,
    description='Offset (frames):',
    style={'description_width': 'auto'}
)

offset_calculation_info_message = widgets.HTML(
    value='<i>The Offset value will be overwritted if "Calculate offset" box is ticked.</i>',
)

checkbox_offset_calculation_input = widgets.Checkbox(
    value=True,
    description='Calculate offset from video rate & beat rate',
    indent=False,
    style={'description_width': 'auto'}
)

offset_fraction_input = widgets.FloatText(
    value = 0.2, 
    description='Fractional offset for automated calculation. ',
    indent = False, 
    style={'description_width': 'auto'}
)

checkbox_calibration_w_pixel_size = widgets.Checkbox(
    value=True,
    description='Calibrate speed/contraction with pixel calibration below?' ,
    indent=False,
    style={'description_width': 'auto'},
    layout=widgets.Layout(width='600px')
)

pixel_calibration_input = widgets.FloatText(
    value=1.3802,
    description='Pixel Calibration (µm/px):',
    style={'description_width': 'auto'}
)

# Run button
run_button = widgets.Button(
    description='Submit parameters',
    button_style='success',
    icon='check'
)

# Output area
output = widgets.Output()

# Define action on button click
def on_run_button_clicked(b):
    with output:
        clear_output()
        directory = input_directory_input.value
        auto_batch = checkbox_batch_processing.value
        file_name_info = file_name_info_message.value
        filename = input_file_input.value
        video_rate = video_rate_input.value
        beat_rate = exp_beat_rate_input.value
        offset = offset_input.value
        offset_calculation_info = offset_calculation_info_message.value
        auto_offset = checkbox_offset_calculation_input.value
        offset_fraction = offset_fraction_input.value
        auto_calibration = checkbox_calibration_w_pixel_size.value
        pixel_calibration = pixel_calibration_input.value
        
        offset = None
        if auto_offset:
            offset = int(offset_fraction_input.value * video_rate / beat_rate)
        else: 
            offset = int(offset_input.value)
        
        print(f"Thanks for initiating the analysis. Run the next cell to analyse the specified dataset.")
run_button.on_click(on_run_button_clicked)

# Display form
form = widgets.VBox([
    input_directory_input,
    checkbox_batch_processing, 
    file_name_info_message, 
    input_file_input,
    video_rate_input,
    exp_beat_rate_input,
    offset_input,
    offset_calculation_info_message, 
    checkbox_offset_calculation_input,
    offset_fraction_input, 
    checkbox_calibration_w_pixel_size, 
    pixel_calibration_input,
    run_button,
    output
])

display(form)

# Analysis and automated result saving

1. **Save analysis parameters**

    - The parameters specified above will be saved automatically in a file called [OriginalFileName_processing_params.txt] in a folder called [analysis] that will be created in the same directory as the file(s) to be processed. This is for accountability and reproducibility purposes. 

2. **Import transmitted light time series image data**

    - In single file analysis mode, once the input data has been imported, the image data will then be displayed in the napari viewer. 

3. **Analysis**

    - The cell below does the analysis and might take some time to run. First, the pixel variance is calculated and from that the speed and contraction data. Please be patient as the processing might take some time depending on the size of the input data and the hardware the notebook is run on.  

4. **Generating speed and contraction plots**

    - The plots are generated and automatically saved in the analysis directory carrying the input file name with appropriate suffixes.

5. **Plot pixel variance with raw input data**

    - The start of the pixel variance stack is padded with black frames according to the set offset parameter. This is to align the raw and pixel variance data for display and overlay purposes. The padded pixel variance stack is then saved as an ome.tiff file
    - In single file mode, the padded pixel variance stack will be displayed in the napari viewer.  

In [2]:
# Collect input and build file list
input_directory = Path(input_directory_input.value).expanduser().resolve()

if not input_directory.is_dir():
    raise ValueError(f"Not a directory: {input_directory}")

def is_ome_tiff(p: Path):
    return p.suffix.lower() in {".tif", ".tiff"} or any(
        str(p).lower().endswith(ext) for ext in [".ome.tif", ".ome.tiff"]
    )

auto_batch = checkbox_batch_processing.value
if auto_batch: 
    files = sorted([p for p in input_directory.iterdir() if p.is_file() and is_ome_tiff(p)])
    print("List of files in the directory that will be processed: ")
    for p in files: 
        print(p.name)
    
    if not files:
        raise RuntimeError("No .tif/.tiff/.ome.tiff files found in the folder.")
else: 
    files = [input_directory / input_file_input.value]

#Create output folder
output_directory = input_directory / "analysis"
output_directory.mkdir(exist_ok=True)
print(f"\nOutput will be written to: {output_directory}\n")

#Process file
def process_file(input_directory, input_file: Path, output_directory: Path): 

    print(f"--- Processing {input_file.name} ---\n")

    input_directory = Path(input_directory_input.value)
    video_rate = video_rate_input.value
    exp_beat_rate = exp_beat_rate_input.value
    offset = int(offset_input.value)
    auto_offset = checkbox_offset_calculation_input.value  # gets overridden next:
    if auto_offset:
        offset = int(offset_fraction_input.value * video_rate / exp_beat_rate)  # calculated, not from widget
    checkbox_offset_calculation = checkbox_offset_calculation_input.value
    auto_calibration = checkbox_calibration_w_pixel_size.value
    pixel_calibration = pixel_calibration_input.value
    
    # print(f"Running with: {input_directory}, {input_file}, {video_rate}, {exp_beat_rate}, {offset}, {checkbox_offset_calculation}")

    input_path = input_directory / input_file

    suffixes = input_file.suffixes  # e.g. ['.ome', '.tiff']
    
    if suffixes == ['.ome', '.tiff']:
        filename_without_extension = input_file.stem.replace(".ome", "")
    else:
        filename_without_extension = input_file.stem
    
    # Write applied processing paraemters to file
    with open(output_directory / f"{filename_without_extension}_processing_params.txt", "w") as f:
        f.write(f"Directory: {output_directory}\n")
        f.write(f"File: {input_file}\n")
        f.write(f"Video rate: {video_rate} fps\n")
        f.write(f"Beat rate: {exp_beat_rate} Hz\n")
        f.write(f"Offset: {offset}\n")
        f.write(f"Auto offset calc: {auto_offset}\n")
        if offset is not None:
            f.write(f"Calculated offset: {offset}\n")
            f.write(f"Offset fraction: {offset_fraction_input.value}\n")
        else:
            f.write("Offset specified manually\n")
        f.write(f"Pixel calibration: {auto_calibration}\n")
        f.write(f"Pixel calibration: {pixel_calibration}\n")
   
    # Load time series image
    print("Importing image data. ...")
    TimeSeriesData = tifffile.imread(input_path)  # shape: (frames, height, width)
    # print(TimeSeriesData.shape)
    TimeSeriesData = TimeSeriesData.astype(np.uint16)
    TimeSeriesData = np.transpose(TimeSeriesData, (1, 2, 0))  # to (H, W, T)
    number_of_frames = TimeSeriesData.shape[2]

    if auto_batch == False: 
        viewer = napari.Viewer()
        viewer.add_image(np.transpose(TimeSeriesData, (2, 0, 1)), name="Original Stack", blending="additive")
    
    
    # ---- First: Pixel Variance over rolling window ----
    print("Calculating Pixel Variance data. ...")
    PV_data_stack = np.zeros((TimeSeriesData.shape[0], TimeSeriesData.shape[1], number_of_frames - offset), dtype=np.float32)
    
    # Fill frames
    for j in range(number_of_frames - offset): 
        PV_data_stack[:, :, j] = np.std(TimeSeriesData[:, :, j:j + offset + 1], axis=2, ddof=1)
    
    # Normalize to 16-bit
    print("Normalising Pixel Variance data. Working hard, this might take some time (and memory) depending on the image size, time series length and computing hardware..")
    pv_min = PV_data_stack.min()
    pv_max = PV_data_stack.max()
    PV_data_stack = np.round((PV_data_stack - pv_min) / (pv_max - pv_min) * 65535).astype(np.uint16)
    
    #print(PV_data_stack.shape)  
    
    # ---- Second: Calculate 'speed' ----
    print("Calculating Speed. More to come but the worst is done ;) ...")
    speed = [np.mean(PV_data_stack[:, :, j]) for j in range(PV_data_stack.shape[2])]
    speed = np.array(speed)

    if auto_calibration: 
        speed_scaled = speed * pixel_calibration
    else: 
        speed_scaled = speed
    # Normalize to [0, 1]
    speed_normalized = (speed_scaled - speed_scaled.min()) / (speed_scaled.max() - speed_scaled.min())
    
    #print(normalized_vec)
    
    # ---- Third: Find key_frame (min speed) ----
    key_frame = np.argmin(speed)
    #print(key_frame)
    
    # ---- Fourth: Calculate contraction ----
    print("Calculating Contraction data. More to come...")
    sample_window = offset
    TimeSeriesData = TimeSeriesData.astype(np.float32)
    contraction = []
    
    for i in range(number_of_frames - sample_window):
        # Standard deviation over the sample window for each pixel
        current_std = np.std(TimeSeriesData[:, :, i:i + sample_window + 1], axis=2, ddof=1)
        key_std = np.std(TimeSeriesData[:, :, key_frame:key_frame + sample_window + 1], axis=2, ddof=1)
        contraction.append(np.mean(np.abs(current_std - key_std)))
    
    contraction = np.array(contraction)
    
    # contraction = 1D NumPy array
    N = len(contraction)
    f_contraction = fftshift(fft(contraction))
    filt = windows.hann(N, sym=False)  # Hann filter
    f_contraction *= filt  # Apply symmetric Hann window
    contraction_filtered = np.real(ifft(fftshift(f_contraction)))  # inverse with fftshift
    if auto_calibration: 
        contraction_scaled = contraction_filtered * pixel_calibration
    else: 
        contraction_scaled = contraction_filtered
    #print(speed_scaled)
    #print(len(contraction), len(contraction_filtered))
    
    # Normalize to [0, 1]
    contraction_normalized = (contraction_scaled - contraction_scaled.min()) / (contraction_scaled.max() - contraction_scaled.min())
    
    # ---- Time vectors and save data ----
    if auto_calibration: 
        time_series_c = np.arange(len(contraction)) / video_rate
        output_c = np.vstack((time_series_c, contraction_filtered, contraction_scaled, contraction_normalized)).T
        
        time_series_s = np.arange(len(speed)) / video_rate
        output_s = np.vstack((time_series_s, speed, speed_scaled, speed_normalized)).T
        # Save data
        np.savetxt(
            output_directory / f"{filename_without_extension}_contraction.txt",
            output_c,
            header="time (s)\tcontraction (px)\tcontraction (µm)\tcontraction (normalized)",
            fmt=("%.4f", "%.8f", "%.8f", "%.8f"),
            delimiter='\t'        )
        np.savetxt(
            output_directory / f"{filename_without_extension}_speed.txt",
            output_s,
            header="time (s)\tspeed (px/s)\tspeed (µm/s)\tspeed (normalized)",
            fmt=("%.4f", "%.8f", "%.8f", "%.8f"),
            delimiter='\t'
        )
        
        print("Done generating and saving results. More to come...")
        
            # ---- Plotting ----
        plt.figure()
        plt.plot(time_series_c, contraction_scaled)
        plt.title('Contraction')
        plt.xlabel('Time (s)')
        plt.ylabel('Contraction (µm)')
        plt.savefig(output_directory / f"{filename_without_extension}_contraction_scaled.png")
        plt.close()
        
        plt.figure()
        plt.plot(time_series_c, contraction_normalized)
        plt.title('Contraction')
        plt.xlabel('Time (s)')
        plt.ylabel('Contraction (normalized)')
        plt.savefig(output_directory / f"{filename_without_extension}_contraction_normalized.png")
        plt.close()
        
        plt.figure()
        plt.plot(time_series_s, speed_scaled)
        plt.title('Speed')
        plt.xlabel('Time (s)')
        plt.ylabel('Speed (µm/s)')
        plt.savefig(output_directory / f"{filename_without_extension}_speed_scaled.png")
        plt.close()
        
        plt.figure()
        plt.plot(time_series_s, speed_normalized)
        plt.title('Speed')
        plt.xlabel('Time (s)')
        plt.ylabel('Speed (normalized)')
        plt.savefig(output_directory / f"{filename_without_extension}_speed_normalized.png")
        plt.close()
        
        print("Done generating and saving plots. More to come...")

    else: 
        time_series_c = np.arange(len(contraction)) / video_rate
        output_c = np.vstack((time_series_c, contraction_filtered, contraction_normalized)).T
        
        time_series_s = np.arange(len(speed)) / video_rate
        output_s = np.vstack((time_series_s, speed, speed_normalized)).T
        # Save data
        np.savetxt(
            output_directory / f"{filename_without_extension}_contraction.txt",
            output_c,
            header="time (s)\tcontraction (px)\tcontraction (normalized)",
            fmt=("%.4f", "%.8f", "%.8f"),
            delimiter='\t'
        )
        np.savetxt(
            output_directory / f"{filename_without_extension}_speed.txt",
            output_s,
            header="time (s)\tspeed (px/s)\tspeed (normalized)",
            fmt=("%.4f", "%.8f", "%.8f"),
            delimiter='\t'
        )
        
        print("Done generating and saving results. More to come...")
        
        # ---- Plotting ----
        plt.figure()
        plt.plot(time_series_c, contraction_scaled)
        plt.title('Contraction')
        plt.xlabel('Time (s)')
        plt.ylabel('Contraction (px)')
        plt.savefig(output_directory / f"{filename_without_extension}_contraction.png")
        plt.close()
        
        plt.figure()
        plt.plot(time_series_c, contraction_normalized)
        plt.title('Contraction')
        plt.xlabel('Time (s)')
        plt.ylabel('Contraction (normalized)')
        plt.savefig(output_directory / f"{filename_without_extension}_contraction_normalized.png")
        plt.close()
        
        plt.figure()
        plt.plot(time_series_s, speed_scaled)
        plt.title('Speed')
        plt.xlabel('Time (s)')
        plt.ylabel('Speed (px/s)')
        plt.savefig(output_directory / f"{filename_without_extension}_speed.png")
        plt.close()
        
        plt.figure()
        plt.plot(time_series_s, speed_normalized)
        plt.title('Speed')
        plt.xlabel('Time (s)')
        plt.ylabel('Speed (normalized)')
        plt.savefig(output_directory / f"{filename_without_extension}_speed_normalized.png")
        plt.close()
    
        print("Done generating and saving plots. More to come...")
    
    # ---- Padding, disolaying and saving Pixel variance data ---- 
    # create the zero padding
    zeros = np.zeros((PV_data_stack.shape[0], PV_data_stack.shape[1], offset), dtype=PV_data_stack.dtype)
    
    # put the zero frames in front
    PV_data_stack = np.concatenate((zeros, PV_data_stack), axis=2)
    
    # Save stack with applied metadata
    if auto_calibration == False:
        pixel_calibration = 1
        pixel_unit = "pixel"
    else: 
        pixel_unit = "micron"
    output_tiff = output_directory / f"{filename_without_extension}_PixelVariance.ome.tiff"
    tifffile.imwrite(output_tiff, 
                     np.transpose(PV_data_stack, (2, 0, 1)), 
                     photometric='minisblack',
                     metadata={
                        "axes": "TYX",  # for example: Time, Y, X
                        "PhysicalSizeX": pixel_calibration,
                        "PhysicalSizeXUnit": pixel_unit,
                        "PhysicalSizeY": pixel_calibration,
                        "TimeIncrement": 1/video_rate,
                    })

    if auto_batch == False: 
        viewer.add_image(np.transpose(PV_data_stack, (2, 0, 1)), name="Pixel Variance Stack", blending="additive", colormap="magenta")

    #Clean up variables to free up memory
    del N, PV_data_stack, TimeSeriesData, checkbox_offset_calculation, contraction, contraction_filtered, contraction_normalized, contraction_scaled, current_std, f, f_contraction, filename_without_extension, filt, i, input_file, input_path, j, key_frame, key_std, number_of_frames, output_c, output_s, output_tiff, pv_max, pv_min, sample_window, speed, speed_normalized, speed_scaled, suffixes, time_series_c, time_series_s, zeros    
    #print(dir())

n = 1
for f in files: 
    print(f"Processing file {n}/{len(files)}.\n")
    process_file(input_directory, f, output_directory)
    n += 1

print("\n*** Full analysis pipeline is finshed. Thank you for your patience. :) ***")


Output will be written to: C:\Users\mheldb\Documents\Image_analysis\2025-07-02-BURTON-Will-python-cardiac-frequency-analysis\input-data\202512-17-ioLight\analysis

Processing file 1/1.

--- Processing tiff_stills.tif ---

Importing image data. ...
Calculating Pixel Variance data. ...
Normalising Pixel Variance data. Working hard, this might take some time (and memory) depending on the image size, time series length and computing hardware..
Calculating Speed. More to come but the worst is done ;) ...
Calculating Contraction data. More to come...
Done generating and saving results. More to come...
Done generating and saving plots. More to come...

*** Full analysis pipeline is finshed. Thank you for your patience. :) ***
